1.start from bytes its demo

In [3]:
text="low lower lowest"
ids=list(text.encode("utf-8"))
print(ids)

[108, 111, 119, 32, 108, 111, 119, 101, 114, 32, 108, 111, 119, 101, 115, 116]


2.counter pair the real tokenizer starts from here

In [10]:
from collections import Counter
def get_pair_count(ids):
    return Counter(zip(ids,ids[1:]))
print(get_pair_count(ids).most_common(3))


[((108, 111), 3), ((111, 119), 3), ((32, 108), 2)]


3. now we gonna merge the repeating tokens with new tokens

In [11]:
def merge(ids,pair,new_id):
    

    out,i=[],0
    while i<len(ids):
        if i<len(ids)-1 and (ids[i],ids[i+1])==pair:
            out.append(new_id)
            i+=2
        else:
            out.append(ids[i])
            i+=1

    return out


4. now we gonna train the bpe

In [13]:
def train_bpe(text,num_train):
    ids=list(text.encode('utf-8'))
    merges={}
    for i in range(num_train):

        counts=get_pair_count(ids)
        if not counts:
            break
        pairs=counts.most_common(1)[0][0]
        new_id=256+i
        ids=merge(ids,pairs,new_id)
        merges[pairs]=new_id
    return merges

merges=train_bpe("bro my name is sijal gharti magar",5)
print(merges)



{(32, 109): 256, (97, 114): 257, (98, 114): 258, (258, 111): 259, (259, 256): 260}


5. now at here we gonna build vocab and ...

In [20]:
def encode(text,merges):
    ids=list(text.encode('utf-8'))
    for pair,new_id in merges.items():
        ids=merge(ids,pair,new_id)
    return ids

encoded_ids=encode("bro",merges)
def vocab(merges):
    vocab={i:bytes([i]) for i in range(256)}
    for (a,b),new_id in merges.items():
        vocab[new_id]=vocab[a]+vocab[b]
    return vocab
vocab=vocab(merges)
def decode(ids,vocab):
    return b"".join(vocab[i] for i in ids).decode('utf-8',errors="reversed")
print(decode(encoded_ids,vocab))

bro
